## Init session

In [ ]:
%load_ext autoreload
%autoreload 2

### Install dependencies

In [ ]:
!chmod +x install.sh
! ./install.sh > /dev/null 2>&1

### Import packages

In [ ]:
import os
import boto3
import subprocess

from pathlib import Path
from random import randint

from rich.pretty import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import torch

from sklearn.model_selection import train_test_split

import model as ic
from datetime import datetime, timedelta

## Download from S3 bucket

In [ ]:
# 1. Automatically retrieve credentials from the Onyxia environment
mybucket = "maximelenormand"
key = os.environ.get('AWS_ACCESS_KEY_ID')
secret = os.environ.get('AWS_SECRET_ACCESS_KEY')
token = os.environ.get('AWS_SESSION_TOKEN')
endpoint = os.environ.get('AWS_ENDPOINT_URL')

# 2. Initialize the client
s3 = boto3.client("s3",endpoint_url = endpoint,
                  aws_access_key_id = key, 
                  aws_secret_access_key = secret, 
                  aws_session_token = token)

# 3. Check connection
listobjbucket = s3.list_objects_v2(Bucket=mybucket, MaxKeys=5)
if 'Contents' in listobjbucket:
    print("Connection OK!\nFiles:")
    for obj in listobjbucket['Contents']:
        print("-", obj['Key'])

In [ ]:
# Download data.zip if needed
#if not os.path.exists("/home/onyxia/work/data.zip"):
#    s3.download_file(mybucket, "data.zip", "data.zip")

# Unzip data.zip if needed
#if not os.path.exists("/home/onyxia/work/data"):
#    subprocess.run(["unzip", "data.zip"],
#                   stdout=subprocess.DEVNULL,
#                   stderr=subprocess.DEVNULL)
#    print("Done!")

# Download mlflow.db if needed
#if not os.path.exists("/home/onyxia/work/mlflow.db"):
#    s3.download_file(mybucket, "mlflow.db", "mlflow.db")

# Download mlruns.zip if needed
#if not os.path.exists("/home/onyxia/work/mlruns.zip"):
#    s3.download_file(mybucket, "mlruns.zip", "mlruns.zip")

# Unzip mlruns.zip if needed
#if not os.path.exists("/home/onyxia/work/mlruns"):
#    subprocess.run(["unzip", "mlruns.zip"],
#                   stdout=subprocess.DEVNULL,
#                   stderr=subprocess.DEVNULL)
#    print("Done!")

## Load & clean data

### Import annotations

In [ ]:
binary_columns = ["human", "anthropic", "vegetation", "rock", "snow"]
annotations = pd.read_csv(Path(".").joinpath("data").joinpath("annotations.csv"), dtype={"image": str})
images = pd.read_csv(Path(".").joinpath("data").joinpath("images.csv"), dtype={"image": str})
tab_raw = pd.concat([annotations.iloc[:, :1],
                     images.iloc[:, 1:2],
                     annotations.iloc[:, 1:],], axis=1)
print(tab_raw)

### Select sites

In [ ]:
sites_trainvaltest = ["carpathians", "french_alps", "stubai_valley", "vinschgau"]
sites_external = ["danube", "dovre", "sierra_nevada"]
sites = sites_trainvaltest + sites_external

tab_external = tab_raw[tab_raw["site"].isin(sites_external)].copy()
tab_raw = tab_raw[tab_raw["site"].isin(sites_trainvaltest)]

### Split TrainVal and Test

In [ ]:
# Split data into train_val vs test before balancing data
def split_data_first(tab, test_size=0.15, random_state=42):
    """
    Split data into train_validation, and test sets BEFORE any balancing.
    The test set remains as a realistic representation of the original distribution.
    """
    # Create stratification column
    tab_strat = tab.copy()
    tab_strat["strat"] = ""
    for col in binary_columns:
        tab_strat["strat"] += tab_strat[col].astype(str)
    
    # First split: train_val vs test (85/15)
    train_val, test = train_test_split(
        tab_strat, 
        test_size=test_size, 
        random_state=random_state, 
        stratify=tab_strat["strat"]
    )
    
    
    # Drop stratification column from final sets
    test = test.drop("strat", axis=1)
    train_val = train_val.drop("strat", axis=1)
    
    return train_val, test

# Perform the split
train_val_raw, test_raw = split_data_first(tab_raw)

### Define a function to rebalance categories

In [ ]:
def rebalance_training_set(train_data, binary_columns, n=2000, seed=42):
    """
    Rebalance only the training set by selecting a balanced subset.
    Validation and test sets remain untouched.
    """
    # Compute weights from imbalance in the training dataframe
    k = 4  # max additional weight on top of the baseline 1
    weights = {}
    for col in binary_columns:
        p = train_data[col].mean()
        imbalance = abs(p - 0.5) / 0.5
        weights[col] = 1 + k * imbalance
    
    # Greedy balanced selection on training data only
    rng = np.random.default_rng(seed)
    df_shuffled = train_data.sample(frac=1, random_state=seed).reset_index(drop=True)
    
    selected_indices = []
    counts = {col: {0: 0, 1: 0} for col in binary_columns}
    target = n // 2
    
    base_tolerance = 500
    ramp = 1000
    
    values = df_shuffled[binary_columns].values
    
    for idx, row_vals in enumerate(values):
        if len(selected_indices) >= n:
            break
        
        score = 0
        for j, col in enumerate(binary_columns):
            val = int(row_vals[j])
            w = weights[col]
            score += w * (
                (target - counts[col][val])
                - (target - counts[col][1 - val])
            )
        
        progress = len(selected_indices) / n
        threshold = -(base_tolerance + progress * ramp)
        
        if score >= threshold:
            selected_indices.append(idx)
            for j, col in enumerate(binary_columns):
                counts[col][int(row_vals[j])] += 1
    
    # Create balanced training set
    train_balanced = df_shuffled.loc[selected_indices].reset_index(drop=True)
    
    return train_balanced

### Define a function to create a control dataset (with the same number of images than the balanced dataset)

In [ ]:
def create_control_set(train_data, train_balanced, seed=123):
    """
    Create a random control set from the training data,
    with the same size as the balanced training set.
    """
    train_control = (
        train_data
        .sample(n=len(train_balanced), random_state=seed)
        .reset_index(drop=True)
    )

    return train_control

### Define a function to compute support by dataset, site, and label

In [ ]:
def compute_support(train_full, train_balanced, train_control, val, seed):
    """
    Create two tables computing the support by dataset and label.
    
    Parameters:
    -----------
    train_full : pd.DataFrame
        The train_full dataframe (before balancing)
    train_blanced : pd.DataFrame
        The train_balanced dataframe
    train_control : pd.DataFrame
        The train_control dataframe
    val :  
        The val dataframe
    seed : int
        Random seed for train/val split 
    
    Returns:
    --------
    pd.DataFrame
        Two tables with support by dataset and label
    """

    # Initialize tables
    support_dataset = []
    support_lable = []

    # Support by site and dataset
    for site in sites:
        row = [(train_full["site"] == site).sum(),
               (train_balanced["site"] == site).sum(),
               (train_control["site"] == site).sum(),
               (val["site"] == site).sum()]
        support_dataset.append(row)

    support_dataset = pd.DataFrame(support_dataset,
                                   index=sites,
                                   columns=["train_full",
                                            "train_balanced",
                                            "train_control",
                                            "val"])
    # Support by label and dataset
    support_label = pd.DataFrame({"train_full": train_full.iloc[:, 2:].sum(),
                                  "train_balanced": train_balanced.iloc[:, 2:].sum(),
                                  "train_control": train_control.iloc[:, 2:].sum(),
                                  "val": val.iloc[:, 2:].sum()})

    support_label = pd.DataFrame(support_label,
                                 index=train_full.iloc[:, 2:].columns,
                                 columns=["train_full",
                                          "train_balanced",
                                          "train_control",
                                          "val"])


    # Save to CSV
    os.makedirs("outputs/tables", exist_ok=True)
    support_dataset.to_csv("outputs/tables/support_dataset_" + str(seed) + ".csv")
    support_label.to_csv("outputs/tables/support_label_" + str(seed) + ".csv")


## Minigrid experiment

In [ ]:
def run_minigrid(train_val, run=True):
    """
    Run the complete experiment pipeline with hyperparameter tuning.
    
    Parameters:
    -----------
    train_val : pd.DataFrame
        The input dataframe containing training and validation data
    run : boolean
        A boolean indicating if the experiment should be run (if False only the support will be computed
    """
    # Set the backbones, repetitions, learning rates and batch size to experiment
    backbones = ["hf_swt_t", "hf_resnet", "hf_cnx2_t", "hf_vit_g16"]
    repetitions = [3, 4, 5] 
    learning_rates = [0.0001, 0.00001]
    batch_sizes = [16, 32]
    
    # Stratify train_val dataset
    tab_strat = train_val.copy()
    tab_strat["strat"] = ""
    for col in train_val.columns[2:]:
        tab_strat["strat"] += tab_strat[col].astype(str)

    # Loop on all parameters
    for rep in repetitions:
     
        # Split in train and val
        train, val = train_test_split(
            train_val,
            test_size=0.18,
            stratify=tab_strat["strat"],
            random_state=rep
        )

        # Create training configurations
        train_balanced = rebalance_training_set(train, binary_columns, n=2000, seed=42)
        train_control = create_control_set(train, train_balanced, seed=123)
        train_full = train.copy()  

        # Compute support
        compute_support(train_full, train_balanced, train_control, val, rep)

        # If run
        if run:
        
            # Loop on all configurations
            for backbone in backbones:
                for lr in learning_rates:
                    for bs in batch_sizes:
                            
                        # Minigrid with full configuration
                        ic.train_model(
                            train_data=train_full,
                            val_data=val,
                            batch_size=bs,
                            max_epochs=20,  
                            image_size=224,
                            run_owner="onyxia",
                            exp_name="minigrid_full",
                            backbone=backbone,
                            loss_name="bce",
                            loss_params={"alpha":0.25, "gamma":2},  
                            device=ic.get_device(),
                            checkpoints_n_saved=1,
                            learning_rate=lr,
                            early_stoper_patience=5,
                            early_stoper_min_delta=0.001,
                            use_lr_finder=False,
                            lr_scheduler_step=10,
                            lr_scheduler_gamma=0.85,
                            print_steps="print",
                            log_progress=False,
                            plot_loss=False,
                            num_workers=10,
                        )

                        # Minigrid with balanced configuration
                        ic.train_model(
                            train_data=train_balanced,
                            val_data=val,
                            batch_size=bs,
                            max_epochs=20,  
                            image_size=224,
                            run_owner="onyxia",
                            exp_name="minigrid_balanced",
                            backbone=backbone,
                            loss_name="bce",
                            loss_params={"alpha":0.25, "gamma":2},  
                            device=ic.get_device(),
                            checkpoints_n_saved=1,
                            learning_rate=lr,
                            early_stoper_patience=5,
                            early_stoper_min_delta=0.001,
                            use_lr_finder=False,
                            lr_scheduler_step=10,
                            lr_scheduler_gamma=0.85,
                            print_steps="print",
                            log_progress=False,
                            plot_loss=False,
                            num_workers=10,
                        )
        
                        # Minigrid with control configuration
                        ic.train_model(
                            train_data=train_control,
                            val_data=val,
                            batch_size=bs,
                            max_epochs=20,  
                            image_size=224,
                            run_owner="onyxia",
                            exp_name="minigrid_control",
                            backbone=backbone,
                            loss_name="bce",
                            loss_params={"alpha":0.25, "gamma":2},  
                            device=ic.get_device(),
                            checkpoints_n_saved=1,
                            learning_rate=lr,
                            early_stoper_patience=5,
                            early_stoper_min_delta=0.001,
                            use_lr_finder=False,
                            lr_scheduler_step=10,
                            lr_scheduler_gamma=0.85,
                            print_steps="print",
                            log_progress=False,
                            plot_loss=False,
                            num_workers=10,
                        )

In [ ]:
# Run minigrid experiment
run_minigrid(train_val_raw, run=False)

### Export minigrid outputs

In [ ]:
def extract_mlflow_results_minigrid(experiment_names, output_file):
    """
    Extract and combine MLflow run results from specified experiments.
    
    Parameters:
    -----------
    experiment_names : list or str
        Name(s) of the MLflow experiment(s) to search
    output_file : str
        Path where the combined results CSV should be saved
    
    Returns:
    --------
    pd.DataFrame
        The combined results dataframe
    """
    # Convert single experiment name to list if needed
    if isinstance(experiment_names, str):
        experiment_names = [experiment_names]
    
    # Search for runs
    runs = mlflow.search_runs(search_all_experiments=True, experiment_names=experiment_names)

    # Extract relevant parameters
    runs_df = runs[[
        "run_id",
        "params.backbone",
        "params.batch_size",
        "params.learning_rate"
    ]].copy()
    runs_df.columns = ["run_id", "backbone", "batch_size", "learning_rate"]
    
    # Collect all classification reports
    all_data = []
    for _, row in runs_df.iterrows():
        run_id = row["run_id"]
        
        try:
            local_path = mlflow.artifacts.download_artifacts(
                run_id=run_id,
                artifact_path="metrics/classification_report.csv"
            )
            
            df_report = pd.read_csv(local_path, sep=";")
            
            df_report["run_id"] = run_id
            df_report["backbone"] = row["backbone"]
            df_report["batch_size"] = row["batch_size"]
            df_report["learning_rate"] = row["learning_rate"]
            
            all_data.append(df_report)
            
        except Exception as e:
            print(f"Error for {run_id}: {e}")
    
    # Combine all results
    final_df = pd.concat(all_data, ignore_index=True)
    
    # Reorder columns
    cols_order = ["run_id", "backbone", "batch_size", "learning_rate"]
    final_df = final_df[
        cols_order + [col for col in final_df.columns if col not in cols_order]
    ]
    
    # Save to CSV
    final_df.to_csv(output_file, index=False)
    
    return final_df

In [ ]:
# Extract minigrid outputs
os.makedirs("outputs", exist_ok=True)
df_minigrid_full = extract_mlflow_results_minigrid(
    experiment_names=["minigrid_full"],
    output_file="outputs/results_minigrid_full.csv"
)
df_minigrid_balanced = extract_mlflow_results_minigrid(
    experiment_names=["minigrid_balanced"],
    output_file="outputs/results_minigrid_balanced.csv"
)
df_minigrid_control = extract_mlflow_results_minigrid(
    experiment_names=["minigrid_control"],
    output_file="outputs/results_minigrid_control.csv"
)

### Train 50 models for hf_swt_t

In [ ]:
def train_models(train_val, backbone, learning_rate, batch_size, run=True):
    """
    Train a model with specified hyperparameters.
    
    Parameters:
    -----------
    train_val : pd.DataFrame
        The input dataframe containing the trainval data 
    backbone : str
        The backbone model to use (e.g., "hf_swt_t", "hf_resnet", "hf_cnx2_t", "hf_vit_g16")
    learning_rate : float
        The learning rate for training (e.g., 0.0001, 0.00001)
    batch_size : int
        The batch size for training (e.g., 16, 32)
    run : boolean
        A boolean indicating if the experiment should be run (if False only the support will be computed
    """
    
    # Stratify train_val dataset
    tab_strat = train_val.copy()
    tab_strat["strat"] = ""
    for col in train_val.columns[2:]:
        tab_strat["strat"] += tab_strat[col].astype(str)

    # Loop over different random seeds for train/val splits
    for rep in [11, 12, 13, 14, 15]:

        # Split in train and val
        train, val = train_test_split(
            train_val,
            test_size=0.18,
            stratify=tab_strat["strat"],
            random_state=rep
        )

        # Create training configurations
        train_balanced = rebalance_training_set(train, binary_columns, n=2000, seed=42)
        train_control = create_control_set(train, train_balanced, seed=123)
        train_full = train.copy()  

        # Compute support
        compute_support(train_full, train_balanced, train_control, val, rep)

        # If run
        if run:

            # Train multiple times with same split (10 repetitions)
            for _ in list(range(10)):
    
                # Training with full configuration
                ic.train_model(
                    train_data=train_full,
                    val_data=val,
                    batch_size=batch_size,  
                    max_epochs=100,
                    image_size=224,
                    run_owner="onyxia",
                    exp_name="swin_full_"+str(rep),  
                    backbone=backbone,  
                    loss_name="bce",
                    loss_params={"alpha": 0.25, "gamma": 2},
                    device=ic.get_device(),
                    checkpoints_n_saved=1,
                    learning_rate=learning_rate,  
                    early_stoper_patience=10,
                    early_stoper_min_delta=0.001,
                    use_lr_finder=False,
                    lr_scheduler_step=10,
                    lr_scheduler_gamma=0.85,
                    print_steps="print",
                    log_progress=False,
                    plot_loss=False,
                    num_workers=10,
                )
    
                # Training with balanced configuration
                ic.train_model(
                    train_data=train_balanced,
                    val_data=val,
                    batch_size=batch_size,  
                    max_epochs=100,
                    image_size=224,
                    run_owner="onyxia",
                    exp_name="swin_balanced_"+str(rep),  
                    backbone=backbone,  
                    loss_name="bce",
                    loss_params={"alpha": 0.25, "gamma": 2},
                    device=ic.get_device(),
                    checkpoints_n_saved=1,
                    learning_rate=learning_rate,  
                    early_stoper_patience=10,
                    early_stoper_min_delta=0.001,
                    use_lr_finder=False,
                    lr_scheduler_step=10,
                    lr_scheduler_gamma=0.85,
                    print_steps="print",
                    log_progress=False,
                    plot_loss=False,
                    num_workers=10,
                )   
    
                # Training with control configuration
                ic.train_model(
                    train_data=train_control,
                    val_data=val,
                    batch_size=batch_size,  
                    max_epochs=100,
                    image_size=224,
                    run_owner="onyxia",
                    exp_name="swin_control_"+str(rep),  
                    backbone=backbone,  
                    loss_name="bce",
                    loss_params={"alpha": 0.25, "gamma": 2},
                    device=ic.get_device(),
                    checkpoints_n_saved=1,
                    learning_rate=learning_rate,  
                    early_stoper_patience=10,
                    early_stoper_min_delta=0.001,
                    use_lr_finder=False,
                    lr_scheduler_step=10,
                    lr_scheduler_gamma=0.85,
                    print_steps="print",
                    log_progress=False,
                    plot_loss=False,
                    num_workers=10,
                ) 

In [ ]:
# Train model 
train_models(
    train_val=train_val_raw,
    backbone="hf_swt_t",
    learning_rate=0.0001,
    batch_size=32,
    run=False
)

### Export trainswt outputs

In [ ]:
def export_mlflow_results_swin(exp_name, output_file, repetitions=None):
    """
    Export MLflow results from experiments where repetition is included in the experiment name.
    
    Parameters:
    -----------
    exp_name : str
        The base name of the experiment (e.g., "trainswt", "trainresnet")
    output_file : str
        Path where the combined results CSV should be saved
    repetitions : list, optional
        List of repetition numbers (default: [11, 12, 13, 14, 15])
    
    Returns:
    --------
    pd.DataFrame
        The combined results dataframe
    """
    if repetitions is None:
        repetitions = [11, 12, 13, 14, 15]
    
    all_data = []
    
    for rep in repetitions:
        
        exp_name_full = f"{exp_name}_{rep}"
        
        exp = mlflow.get_experiment_by_name(exp_name_full)
        
        if exp is None:
            print(f"Warning: Experiment '{exp_name_full}' not found. Skipping...")
            continue
        runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])
        
        for _, row in runs.iterrows():
            run_id = row["run_id"]
            
            try:
                local_path = mlflow.artifacts.download_artifacts(
                    run_id=run_id,
                    artifact_path="metrics/classification_report.csv"
                )
                
                df_report = pd.read_csv(local_path, sep=";")
                
                df_report["run_id"] = run_id
                df_report["rep"] = rep
                
                all_data.append(df_report)
                
            except Exception as e:
                print(f"Error for {run_id}: {e}")
    
    if not all_data:
        print(f"No data found for experiments with base name: {exp_name}")
        return None
    
    final_df = pd.concat(all_data, ignore_index=True)
    
    cols_order = ["run_id", "rep"]
    final_df = final_df[
        cols_order + [col for col in final_df.columns if col not in cols_order]
    ]
    
    final_df.to_csv(output_file, index=False)
    
    return final_df

In [ ]:
# Export trainswt outputs
results_swin_full = export_mlflow_results_swin(
    exp_name="swin_full",
    output_file="outputs/results_swin_full.csv",
    repetitions=[11, 12, 13, 14, 15]
)
results_swin_balanced = export_mlflow_results_swin(
    exp_name="swin_balanced",
    output_file="outputs/results_swin_balanced.csv",
    repetitions=[11, 12, 13, 14, 15]
)
results_swin_balanced = export_mlflow_results_swin(
    exp_name="swin_control",
    output_file="outputs/results_swin_control.csv",
    repetitions=[11, 12, 13, 14, 15]
)

### Find best model

In [ ]:
# Function to get the id of the best model
def get_best_model_id(exp_name):
    """
    Get the run_id of the best model from the corresponding
    exported CSV.

    Parameters:
    -----------
    exp_name : str
        Experiment name: "full", "balanced" or "control".

    Returns:
    --------
    str
        run_id of the best model.
    """

    output_file = os.path.join(
        "outputs",
        f"results_{exp_name}.csv"
    )

    df = pd.read_csv(output_file, sep=",")

    df_weighted = df[
        df["labels"] == "weighted avg"
    ]

    best = df_weighted.loc[
        df_weighted["f1-score"].idxmax()
    ]

    best_model = []
    best_model.append({
        "experiment": exp_name,
        "best_model_id": best["run_id"],
        "rep": best["rep"],
        "f1-score": best["f1-score"]
    })

    results = pd.DataFrame(best_model)
    results.to_csv(os.path.join("outputs", f"best_model_{exp_name}.csv"), index=False)

    print(
        f"Best {exp_name}: "
        f"model={best['run_id']} | "
        f"rep={best['rep']} | "
        f"F1={best['f1-score']:.4f}"
    )

    return best["run_id"]



In [ ]:
# Export best model info
best_run_id_full = get_best_model_id(exp_name = "swin_full")
best_run_id_balanced = get_best_model_id(exp_name = "swin_balanced")
best_run_id_control = get_best_model_id(exp_name = "swin_control")

### Export training outputs

In [ ]:
def export_metric_history(best_run_id, filename):
    """
    Export metric history for a given MLflow run to a CSV file.
    
    Args:
        best_run_id (str): The MLflow run ID to fetch metrics for
        filename (str): The name/path of the file to export the CSV to
    """
    client = mlflow.tracking.MlflowClient()
    
    train_loss = client.get_metric_history(best_run_id, "training Loss")
    val_loss   = client.get_metric_history(best_run_id, "validation Loss")
    
    train_f1 = client.get_metric_history(best_run_id, "training F1")
    val_f1   = client.get_metric_history(best_run_id, "validation F1")
    
    
    train_loss_df = pd.DataFrame([
        {"epoch": m.step, "train_loss": m.value}
        for m in train_loss
    ])
    
    val_loss_df = pd.DataFrame([
        {"epoch": m.step, "val_loss": m.value}
        for m in val_loss
    ])
    
    train_f1_df = pd.DataFrame([
        {"epoch": m.step, "train_f1": m.value}
        for m in train_f1
    ])
    
    val_f1_df = pd.DataFrame([
        {"epoch": m.step, "val_f1": m.value}
        for m in val_f1
    ])
    
    df_epoch = (
        train_loss_df
        .merge(val_loss_df, on="epoch", how="outer")
        .merge(train_f1_df, on="epoch", how="outer")
        .merge(val_f1_df, on="epoch", how="outer")
        .sort_values("epoch")
    )
    
    df_epoch.to_csv(filename, index=False)
    
    return df_epoch


In [ ]:
# Export metric history
metric_history_balanced = export_metric_history(
    best_run_id_balanced, "outputs/best_model_curves_balanced.csv")
metric_history_control = export_metric_history(
    best_run_id_control, "outputs/best_model_curves_control.csv")
metric_history_full = export_metric_history(
    best_run_id_full, "outputs/best_model_curves_full.csv")

## Validation

### Export best model predictions

In [ ]:
def export_outputs_bestmodel(test, tab_external, best_run_id, name): 

    # Load best model
    model = mlflow.pytorch.load_model(
        f"runs:/{best_run_id}/model",
        map_location=torch.device(ic.get_device()),
    )

    # Export tresholds
    pd.DataFrame(model.thresholds).to_csv(f"outputs/thresholds_{name}.csv", index=False)

    # Export test predictions
    os.makedirs("outputs/test", exist_ok=True)
    proba =  pd.DataFrame(model.predict_propabilities(dataset=ic.FldDataset(data=test, train_mode=False)))
    proba.to_csv(f"outputs/test/test_proba_{name}.csv", index=False)
    testout = model.get_val_data(dataset=ic.FldDataset(data=test, train_mode=False))
    testout["predictions_revue"].to_csv(f"outputs/test/prediction_revue_{name}.csv", index=False)
    testout["classification_report"].to_csv(f"outputs/test/classification_report_{name}.csv", index=False)

    # Export external predictions
    os.makedirs("outputs/external", exist_ok=True)
    proba =  pd.DataFrame(model.predict_propabilities(dataset=ic.FldDataset(data=tab_external, train_mode=False)))
    proba.to_csv(f"outputs/external/external_proba_{name}.csv", index=False)
    externalout = model.get_val_data(dataset=ic.FldDataset(data=tab_external, train_mode=False))
    externalout["predictions_revue"].to_csv(f"outputs/external/prediction_revue_{name}.csv", index=False)
    externalout["classification_report"].to_csv(f"outputs/external/classification_report_{name}.csv", index=False)


In [ ]:
# Export best model results 
export_outputs_bestmodel(test = test_raw, tab_external = tab_external, 
                         best_run_id = best_run_id_full, name = "full")
export_outputs_bestmodel(test = test_raw, tab_external = tab_external, 
                         best_run_id = best_run_id_balanced, name = "balanced")
export_outputs_bestmodel(test = test_raw, tab_external = tab_external, 
                         best_run_id = best_run_id_control, name = "control")

### Export external predictions per site

In [ ]:
def export_predictions_persite(tab, sites, best_run_id, name): 

    """
    Export the predictions of a given model (best_run_id) for each site separately
    
    Args:
        tab (pd.DataFrame): The input dataframe containing the data
        best_run_id (str): The MLflow run ID to fetch metrics for
        sites (list): Name of all the sites to predict
        name (str): name to include at the end of the file saved ("full" or "balanced")
        
    """
    
    # Load best model
    model = mlflow.pytorch.load_model(
        f"runs:/{best_run_id}/model",
        map_location=torch.device(ic.get_device()),
    )

    # Loop on all sites
    for site in sites:

        # Create directory for each site if needed
        out_dir = f"outputs/sites/{site}"
        os.makedirs(out_dir, exist_ok=True)

        # Subset the dataset for the site
        subset = tab[tab["site"] == site].reset_index(drop=True)
    
        # Convert into FldDataset
        dataset = ic.FldDataset(data=subset, train_mode=False)

        # probabilities
        proba = pd.DataFrame(model.predict_propabilities(dataset=dataset))
        proba.to_csv(f"{out_dir}/proba_{name}.csv", index=False)

        # model outputs
        challengeout = model.get_val_data(dataset=dataset)
        challengeout["predictions_revue"].to_csv(
            f"{out_dir}/prediction_revue_{name}.csv",
            index=False
        )
        challengeout["classification_report"].to_csv(
            f"{out_dir}/classification_report_{name}.csv",
            index=False
        )

In [ ]:
# Export predictions
export_predictions_persite(tab = test_raw, sites = sites_trainvaltest, best_run_id = best_run_id_full, name = "full")
export_predictions_persite(tab = test_raw, sites = sites_trainvaltest, best_run_id = best_run_id_balanced, name = "balanced")
export_predictions_persite(tab = test_raw, sites = sites_trainvaltest, best_run_id = best_run_id_control, name = "control")

export_predictions_persite(tab = tab_external, sites = sites_external, best_run_id = best_run_id_full, name = "full")
export_predictions_persite(tab = tab_external, sites = sites_external, best_run_id = best_run_id_balanced, name = "balanced")
export_predictions_persite(tab = tab_external, sites = sites_external, best_run_id = best_run_id_control, name = "control")

### Prepare files for export

In [ ]:
# Zip mlruns and outputs
subprocess.run(["zip", "-rq", "outputs.zip", "outputs"], check=True)
#subprocess.run(["zip", "-rq", "mlruns.zip", "mlruns"], check=True)

# Upload outputs.zip, mlflow.db and mlruns.zip 
#s3.upload_file("outputs.zip", mybucket, "outputs.zip")
#s3.upload_file("mlflow.db", mybucket, "mlflow.db")
#s3.upload_file("mlruns.zip", mybucket, "mlruns.zip")